[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/07_logistic_regression/06_softmax_regression.ipynb)

# Bài 06 — Phân loại nhiều lớp: Softmax Regression

## 1. Giới thiệu

Bài trước dùng Logistic Regression cho nhiều lớp theo kiểu One-vs-All: các classifier được train **riêng rẽ**, không liên hệ với nhau, điểm số không tạo thành phân phối xác suất.

<img src="images/softmax/onevsrest.png" style="width:55%;height:55%;">
<center>Logistic Regression dùng One-vs-Rest (nguồn: machinelearningcoban.com)</center>

**Softmax Regression** (multinomial logistic regression) thay sigmoid bằng hàm **softmax**: chuẩn hoá $K$ điểm số $z_k$ thành một **phân phối xác suất** (không âm, tổng bằng 1), rồi chọn lớp có xác suất cao nhất. Biểu diễn dưới dạng neural network một tầng:

<img src="images/softmax/softmax_nn.png" style="width:55%;height:55%;">
<center>Softmax Regression (nguồn: machinelearningcoban.com)</center>

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/07_logistic_regression

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from ml_course.visualization import set_style

set_style()

## 2. Mô hình Softmax Regression

### 2.1 Hàm softmax

Mỗi lớp $k$ có vector trọng số $\mathbf w_k$; điểm số (logit) $z_k = \mathbf w_k^\top\mathbf x + b_k$. Xác suất:

$$a_k = P(y = k \mid \mathbf x) = \frac{\exp(z_k)}{\sum_{j=1}^{C}\exp(z_j)}$$

trong đó $0 \le a_k \le 1$, $\sum_k a_k = 1$, $C$ là số lớp.

> 📝 *Đính chính:* mẫu số là $\sum_j \exp(z_j)$ (bản gốc ghi nhầm chỉ số thành $z_i$).

In [ ]:
def softmax_naive(z):
    e_z = np.exp(z)
    return e_z / np.sum(e_z)


data = np.asarray([2.4, -0.5, 3.2, 1.9])
print(softmax_naive(data).round(4), "tổng =", softmax_naive(data).sum())

Ví dụ đầu vào và ra của hàm softmax:

<img src="images/softmax/softmax.png" style="width:35%;height:35%;">

**Hạn chế:** hàm mũ có thể **tràn số** khi $z$ lớn:

In [ ]:
with np.errstate(over="ignore", invalid="ignore"):
    print(softmax_naive(np.asarray([1112.4, -0.5, 3.2, 1.9])))      # overflow → nan

Khắc phục: trừ đi một hằng số $c$ (thường là $\max_j z_j$) — kết quả **không đổi** vì $\frac{e^{z_k - c}}{\sum_j e^{z_j - c}} = \frac{e^{-c}e^{z_k}}{e^{-c}\sum_j e^{z_j}}$:

$$a_k = \frac{\exp(z_k - c)}{\sum_{j=1}^{C}\exp(z_j - c)}$$

In [ ]:
def softmax(Z):
    """Softmax theo CỘT (mỗi cột là một mẫu) — giữ quy ước của bản gốc.
    axis=0: tính theo cột; keepdims=True giữ số chiều sau khi lấy max/sum."""
    e_Z = np.exp(Z - np.max(Z, axis=0, keepdims=True))
    return e_Z / np.sum(e_Z, axis=0, keepdims=True)


data = np.array([[1000, 2, 3, 6],
                 [2, 4, 5, 6],
                 [3, 8, 7, 6]])
print(softmax(data).round(4))

> 💡 Khoá học (và scikit-learn, PyTorch) dùng quy ước **mỗi hàng là một mẫu** → softmax theo `axis=1` (`ml_course.utils.softmax`). Chỉ cần nhất quán.

### 2.2 Cost function — Cross-entropy *(hoàn thiện — bản gốc để trống)*

Mã hoá nhãn bằng **one-hot**: $\mathbf y^{(i)} = [0, \dots, 1, \dots, 0]$ (1 ở vị trí lớp đúng). Likelihood của một mẫu là xác suất model gán cho **lớp đúng**: $\prod_k (a_k^{(i)})^{y_k^{(i)}}$. Lấy log âm, trung bình → **categorical cross-entropy**:

$$J(W) = -\frac{1}{m}\sum_{i=1}^{m}\sum_{k=1}^{C} y^{(i)}_k\log a^{(i)}_k = -\frac{1}{m}\sum_{i=1}^{m}\log a^{(i)}_{y^{(i)}}$$

— chỉ còn log-xác suất của lớp đúng. Với $C = 2$, công thức này trở thành đúng binary cross-entropy của Logistic Regression.

### 2.3 Gradient

Kết quả đẹp như Logistic Regression ("dự đoán − thật"):

$$\frac{\partial J}{\partial \mathbf z^{(i)}} = \frac1m(\mathbf a^{(i)} - \mathbf y^{(i)}) \qquad\Rightarrow\qquad \nabla_W J = \frac{1}{m}X^\top(A - Y), \qquad \nabla_{\mathbf b} J = \frac1m\sum_i(\mathbf a^{(i)} - \mathbf y^{(i)})$$

với $X$ là $(m \times n)$, $W$ là $(n \times C)$, $A = \text{softmax}(XW + \mathbf b)$ và $Y$ là one-hot $(m \times C)$.

### 2.4 Softmax với C = 2 chính là Logistic Regression *(bổ sung)*

$$a_1 = \frac{e^{z_1}}{e^{z_0} + e^{z_1}} = \frac{1}{1 + e^{-(z_1 - z_0)}} = \sigma(z_1 - z_0)$$

— sigmoid của **hiệu** hai logit. Logistic Regression là trường hợp đặc biệt của Softmax Regression.

## 3. Cài đặt from scratch trên digits *(bổ sung)*

In [ ]:
from sklearn.model_selection import train_test_split

from ml_course.datasets import load_digits_data
from ml_course.utils import numerical_gradient, one_hot, relative_error
from ml_course.utils import softmax as softmax_rows

X, y, _ = load_digits_data()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)
C = 10
Y_train = one_hot(y_train, C)


def cost_and_grad(W, b, X, Y, lam=0.0):
    m = len(X)
    A = softmax_rows(X @ W + b)                                        # (m, C)
    J = -np.mean(np.sum(Y * np.log(A + 1e-12), axis=1)) + lam / (2 * m) * np.sum(W**2)
    dZ = (A - Y) / m
    return J, X.T @ dZ + lam / m * W, dZ.sum(axis=0)


# Gradient check trên một tập nhỏ
W0 = np.random.default_rng(0).normal(0, 0.01, (64, C))
b0 = np.zeros(C)
_, dW, _ = cost_and_grad(W0, b0, X_train[:20], Y_train[:20])
num = numerical_gradient(lambda w: cost_and_grad(w.reshape(64, C), b0, X_train[:20], Y_train[:20])[0], W0.ravel().copy())
print("gradient check, relative error:", f"{relative_error(dW.ravel(), num):.2e}")

In [ ]:
def train_softmax(X, Y, lr=0.5, epochs=300, lam=1.0):
    W, b = np.zeros((X.shape[1], Y.shape[1])), np.zeros(Y.shape[1])
    history = []
    for _ in range(epochs):
        J, dW, db = cost_and_grad(W, b, X, Y, lam)
        W -= lr * dW
        b -= lr * db
        history.append(J)
    return W, b, history


W, b, history = train_softmax(X_train, Y_train)
pred = np.argmax(softmax_rows(X_test @ W + b), axis=1)
print(f"Accuracy test (softmax from scratch): {np.mean(pred == y_test):.4f}")
plt.plot(history)
plt.xlabel("epoch")
plt.ylabel("cross-entropy")
plt.show()

Xác suất dự đoán của vài ảnh test — bây giờ **tổng bằng 1**:

In [ ]:
P = softmax_rows(X_test[:3] @ W + b)
print(P.round(3))
print("tổng mỗi hàng:", P.sum(axis=1))

## 4. So sánh với scikit-learn và One-vs-All

In [ ]:
from sklearn.linear_model import LogisticRegression

from ml_course.algorithms import SoftmaxRegressionScratch

ref = SoftmaxRegressionScratch(lr=0.5, n_iters=300, l2=1.0).fit(X_train, y_train)
sk = LogisticRegression(max_iter=3000).fit(X_train, y_train)          # multinomial mặc định
print(f"Softmax (bài này)          : {np.mean(pred == y_test):.4f}")
print(f"SoftmaxRegressionScratch   : {np.mean(ref.predict(X_test) == y_test):.4f}")
print(f"sklearn LogisticRegression : {sk.score(X_test, y_test):.4f}")

Softmax Regression (trên pixel thô) đạt ~95–97% trên digits. Đây là một model **tuyến tính** — ranh giới giữa hai lớp bất kỳ là siêu phẳng. Neural network (chương 27–29) đặt **chính lớp softmax này** ở đầu ra, nhưng thêm các tầng ẩn phi tuyến phía trước để học feature tốt hơn.

## 🧠 Kiểm tra nhanh

1. Vì sao trừ $\max z$ trước khi lấy `exp` không làm thay đổi kết quả softmax?
2. Cross-entropy của một mẫu khi model gán xác suất 0.25 cho lớp đúng là bao nhiêu?
3. Chứng minh softmax với 2 lớp tương đương sigmoid.
4. Số tham số của Softmax Regression cho ảnh 8×8 và 10 lớp?

## Tài liệu tham khảo

[1] [Softmax Regression — Machine Learning Cơ Bản](https://machinelearningcoban.com/2017/02/17/softmax/) · [2] [CS231n: Softmax classifier](https://cs231n.github.io/linear-classify/#softmax)

---
<sub>Bài gốc: *Softmax Regression* — [AIVN-Machine-Learning](https://github.com/thanhhff/AIVN-Machine-Learning), Tuần 5 (MIT License). Đã đính chính công thức softmax; hoàn thiện phần Cost Function (bản gốc để trống), bổ sung gradient, liên hệ với sigmoid, cài đặt from scratch có gradient check và so sánh.</sub>